# Investigación Operativa — Laboratorio 0
## De una formulación matemática a una recomendación reproducible

**Clase 01 · Licenciatura en Ciencia de Datos · Universidad Austral**

### Objetivos

Al finalizar el laboratorio podrá:

1. representar un modelo lineal con la API de `scipy.optimize.linprog`;
2. interpretar el estado del solver antes de usar la solución;
3. verificar restricciones, holguras y valor objetivo;
4. comparar políticas y documentar una recomendación;
5. distinguir una solución computacional de un modelo validado.

> El caso es didáctico y simulado. Los puntos de valor no describen operaciones reales de la Universidad.

## Consigna del taller del martes

### Situación problemática

Un equipo de analítica planifica su producción para la próxima semana. Puede entregar **tableros operativos** y **modelos predictivos**. Cada unidad genera puntos de valor, pero consume horas de CPU, GPU y trabajo de analista. La dirección necesita una recomendación cuantitativa que sea factible, verificable y comprensible; no alcanza con informar lo que devuelve un solver.

El archivo `../Datos/caso_capacidad_analitica.csv` contiene los coeficientes del caso. Los datos son simulados y las variables se modelarán inicialmente como continuas.

### Encargo

Determine cuántas unidades semanales de cada servicio conviene producir para maximizar el valor total. Identifique qué recursos limitan la decisión y analice si una modificación de la capacidad de CPU cambia la recomendación.

### Trabajo obligatorio

1. Identifique decisor, horizonte, variables, unidades, objetivo, restricciones y al menos un supuesto que deba validarse.
2. Audite el CSV y construya el modelo lineal sin copiar coeficientes de la salida del solver.
3. Resuelva con HiGHS, registre el estado antes de leer la solución y verifique independientemente no negatividad, consumos, holguras, violación máxima y objetivo.
4. Represente la región factible y marque la solución; el gráfico debe usar las mismas unidades y coeficientes que el modelo.
5. Evalúe las cuatro políticas propuestas y explique por qué una alternativa de gran valor aparente puede ser inadmisible.
6. Reoptimice para 65, 80 y 90 horas de CPU e interprete los cambios y las restricciones activas.

### Entregables

- Notebook ejecutada de principio a fin, sin errores y con controles automáticos aprobados.
- Tabla de decisión, tabla de uso de recursos, gráfico del conjunto factible y tabla de escenarios.
- Recomendación ejecutiva de 120 a 180 palabras y respuestas individuales del *exit ticket*.

### Criterios de aceptación

La entrega debe conservar unidades, declarar la tolerancia numérica, distinguir solución computacional de modelo validado y conectar cada afirmación con evidencia reproducible. Una captura de pantalla o la salida sin interpretar del solver no constituyen una respuesta.

## Hoja de ruta del taller — 120 minutos

| Minutos | Trabajo | Evidencia mínima |
|---:|---|---|
| 0–10 | Recuperar conceptos y anticipar el resultado | Hipótesis escrita antes de ejecutar |
| 10–25 | Auditar datos, unidades y supuestos | Controles de entrada visibles |
| 25–60 | Construir y resolver el caso base | Estado, solución y valor |
| 60–80 | Verificar por una vía independiente | Violación/error y prueba de consistencia |
| 80–105 | Extender o perturbar el caso | Tabla o gráfico comparativo |
| 105–115 | Auditoría cruzada | Lista de observaciones corregidas |
| 115–120 | Recomendación individual | Decisión, evidencia, límite y próximo dato |

**Regla de trabajo.** Si el resultado contradice la anticipación, no se modifica el código hasta escribir una explicación posible y un control que permita distinguir entre error y fenómeno real.

## Preparación individual — 10 minutos

Sin ejecutar código, anticipe la solución del caso de capacidad analítica. Indique qué dos restricciones espera activas, proponga un punto factible y otro inviable, y escriba el control independiente que detectaría un intercambio accidental entre las columnas CPU y GPU.

### Su respuesta

Escriba aquí su anticipación, dos supuestos críticos y el control que usaría para detectar el error más peligroso.

## Ficha cerrada de datos y supuestos

- Horizonte: una semana; variables continuas y no negativas.
- Tablero operativo: valor 40 puntos; consume 2 h de CPU, 0 h de GPU y 1 h de analista.
- Modelo predictivo: valor 100 puntos; consume 1 h de CPU, 2 h de GPU y 1 h de analista.
- Capacidades: 80 h de CPU, 60 h de GPU y 50 h de analista.
- Objetivo: maximizar puntos de valor semanal. Los coeficientes también están en `../Datos/caso_capacidad_analitica.csv`.
- Supuesto: existe demanda para toda unidad recomendada y los consumos son proporcionales.

## 0. Identificación inicial de la decisión

Un equipo analítico debe asignar su capacidad semanal entre **tableros operativos** y **modelos predictivos**. Dispone de 80 h de CPU, 60 h de GPU y 50 h de analista.

**Pregunta:** ¿cuántas unidades de cada servicio conviene realizar para maximizar el valor total, y qué recursos limitan esa decisión?

Antes de ejecutar código, complete:

- Decisor:
- Horizonte:
- Variables con unidades:
- Audiencia de la recomendación:
- Un supuesto que debería validarse:

In [ ]:
from pathlib import Path
import platform

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy
from scipy.optimize import linprog

print("Python:", platform.python_version())
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("SciPy:", scipy.__version__)

## 1. Datos, unidades y controles

Lea el archivo `../Datos/caso_capacidad_analitica.csv`. Verifique:

- que existan dos servicios;
- que los coeficientes de recursos sean no negativos;
- que las capacidades tengan unidades compatibles con los consumos;
- que no haya valores faltantes donde el modelo necesita un coeficiente.

In [ ]:
DATA = Path("../Datos/caso_capacidad_analitica.csv")
df = pd.read_csv(DATA)
df

In [ ]:
# TODO: separe las filas de servicios de la fila de capacidades.
# Construya una tabla compacta para auditar los coeficientes.

servicios = None
capacidades = None

print("Complete esta celda antes de continuar.")

## 2. Formulación

Defina:

- $x_T$: unidades semanales de tableros operativos;
- $x_M$: unidades semanales de modelos predictivos.

El modelo es:

\begin{aligned}
\max\quad &40x_T+100x_M\\
\text{s.a.}\quad &2x_T+x_M\le80,\\
&2x_M\le60,\\
&x_T+x_M\le50,\\
&x_T,x_M\ge0.
\end{aligned}

`linprog` minimiza. Por lo tanto, para maximizar $c^Tx$ debe minimizar $-c^Tx$.

In [ ]:
# TODO: complete los coeficientes usando el orden [x_T, x_M].
c_max = np.array([np.nan, np.nan])
A_ub = np.array([
    [np.nan, np.nan],  # CPU
    [np.nan, np.nan],  # GPU
    [np.nan, np.nan],  # analista
])
b_ub = np.array([np.nan, np.nan, np.nan])
bounds = [(0, None), (0, None)]

### Control previo

Antes de llamar al solver, explique en una oración qué representa cada fila de `A_ub` y qué unidad tiene el producto matricial `A_ub @ x`.

In [ ]:
# TODO: quite el comentario cuando los coeficientes estén completos.
# resultado = linprog(c=-c_max, A_ub=A_ub, b_ub=b_ub,
#                     bounds=bounds, method="highs")
# print(resultado.message)

## 3. Estado antes que solución

Registre:

- `resultado.success`;
- `resultado.status`;
- `resultado.message`;
- el método utilizado.

No interprete `resultado.x` si el estado no permite hacerlo.

In [ ]:
# TODO: construya un resumen explícito del estado del solver.
# estado = {
#     "success": resultado.success,
#     "status": resultado.status,
#     "message": resultado.message,
# }
# estado

## 4. Verificación automática

Implemente una función que reciba una solución y compruebe:

1. no negatividad;
2. consumo de cada recurso;
3. violación máxima con tolerancia;
4. valor objetivo recalculado;
5. holguras.

In [ ]:
def verificar_solucion(x, A, b, c, tol=1e-7):
    'Devuelva un diccionario con controles de factibilidad y objetivo.'
    # TODO: implementar sin depender del valor objetivo informado por el solver.
    raise NotImplementedError

In [ ]:
# TODO: use la función sobre la solución del solver.
# verificacion = verificar_solucion(resultado.x, A_ub, b_ub, c_max)
# verificacion

## 5. Interpretación

Construya una tabla con:

- variable;
- valor óptimo;
- unidad;
- interpretación operacional.

Luego identifique qué restricciones están activas usando una tolerancia explícita.

In [ ]:
# TODO: tabla de decisión y tabla de recursos.

## 6. Visualización del conjunto factible

Grafique las tres fronteras, sombree la región factible y marque la solución encontrada. El gráfico es una ayuda de validación: debe usar las mismas unidades y coeficientes que la formulación.

In [ ]:
# TODO: construya la visualización para 0 <= x_T <= 45.

## 7. Comparación de políticas

Evalúe, sin volver a optimizar, las políticas:

- A: $(20,20)$;
- B: $(40,0)$;
- C: $(25,30)$;
- D: $(20,30)$.

Para cada una informe factibilidad, valor y recursos violados. Explique por qué una política con valor aparente alto puede ser inadmisible.

In [ ]:
politicas = {
    "A": np.array([20.0, 20.0]),
    "B": np.array([40.0, 0.0]),
    "C": np.array([25.0, 30.0]),
    "D": np.array([20.0, 30.0]),
}

# TODO: evaluar todas las políticas con la función de verificación.

## 8. Experimento: una restricción inactiva

Aumente la capacidad de CPU de 80 a 90 horas y resuelva nuevamente.

1. ¿Cambia la decisión?
2. ¿Cambia el valor óptimo?
3. ¿Cómo se relaciona el resultado con la holgura del modelo base?

Después disminuya CPU a 65 horas y repita. Documente los dos escenarios en una tabla.

In [ ]:
# TODO: función resolver(cap_cpu) y tabla para CPU = 65, 80 y 90.

## 9. Recomendación ejecutiva

Escriba entre 120 y 180 palabras. Debe incluir:

- decisión recomendada y horizonte;
- valor esperado con unidad;
- recursos activos y holguras;
- supuesto crítico;
- límite del modelo;
- dato o experimento necesario antes de implementar.

### Recomendación

_Escriba aquí._

## 10. Exit ticket individual

1. ¿Qué diferencia hay entre `resultado.success` y haber validado el modelo?
2. ¿Qué error detectaría el control dimensional que el solver no detecta?
3. Si los servicios deben entregarse en unidades completas, ¿qué parte del modelo debe cambiar?

## Rutas posibles para explorar

Elija al menos una ruta principal y use otra como control independiente. No se exige implementar las tres durante el taller.

1. solver HiGHS.
2. enumeración exacta de vértices.
3. búsqueda sobre una malla como control aproximado.

Antes de programar, anticipe cuál debería ser exacta, cuál produce una cota o aproximación y qué evidencia permitiría compararlas.

## Extensión guiada — 25 minutos

Extienda el caso de capacidad analítica incorporando una exigencia de al menos 10 tableros, $x_T\ge10$. Resuelva nuevamente, compare con el caso base, verifique todas las restricciones y explique si la nueva regla cambia la decisión, el valor o sólo la descripción del conjunto factible.

### Desarrollo

Desarrolle aquí la extensión. Debe conservar datos, resultado, control e interpretación en bloques separados.

## Auditoría cruzada — 10 minutos

Intercambie la notebook con otra persona y deje evidencia de estas cinco comprobaciones:

1. los datos usados coinciden con la ficha y conservan unidades;
2. el estado del método se lee antes de interpretar valores;
3. objetivo, restricciones o métricas se recalculan fuera del objeto principal;
4. el gráfico o la tabla responde a la misma versión del modelo;
5. la recomendación distingue resultado, supuesto y límite.

Registre al menos una observación recibida y la modificación realizada.

## Lista de control antes de entregar

- [ ] La pregunta de decisión aparece respondida en una frase.
- [ ] Cada variable y cada métrica tiene unidad y horizonte.
- [ ] El código puede ejecutarse desde el inicio sin intervención manual oculta.
- [ ] Hay evidencia de estado, factibilidad o validez estadística, según corresponda.
- [ ] Se compara al menos un escenario o método alternativo.
- [ ] La recomendación incluye una limitación y el próximo dato necesario.